In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mdnaiemhossain03/bank-fraud-dataset/bank_fraud.csv


In [2]:
import pandas as pd

file_path = "/kaggle/input/datasets/mdnaiemhossain03/bank-fraud-dataset/bank_fraud.csv"

df = pd.read_csv(file_path)

print(df.shape)

(1000000, 26)


In [3]:
df.columns.tolist()

['transaction_id',
 'customer_id',
 'transaction_date',
 'transaction_time',
 'hour_of_day',
 'is_weekend',
 'is_night_transaction',
 'country',
 'city',
 'merchant_category',
 'payment_method',
 'device_type',
 'customer_age',
 'credit_score',
 'account_age_years',
 'account_balance',
 'transaction_amount',
 'num_prev_transactions',
 'transaction_freq_monthly',
 'distance_from_home_km',
 'time_since_last_txn_hrs',
 'is_international',
 'failed_attempts',
 'pin_changed_recently',
 'is_fraud',
 'fraud_type']

In [4]:
TARGET = "is_fraud"

print(df[TARGET].value_counts(dropna=False))

is_fraud
0    944745
1     55255
Name: count, dtype: int64


In [5]:
df.isnull().sum()

transaction_id                   0
customer_id                      0
transaction_date                 0
transaction_time                 0
hour_of_day                      0
is_weekend                       0
is_night_transaction             0
country                          0
city                             0
merchant_category                0
payment_method                   0
device_type                      0
customer_age                     0
credit_score                     0
account_age_years                0
account_balance                  0
transaction_amount               0
num_prev_transactions            0
transaction_freq_monthly         0
distance_from_home_km            0
time_since_last_txn_hrs          0
is_international                 0
failed_attempts                  0
pin_changed_recently             0
is_fraud                         0
fraud_type                  944745
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.dtypes

transaction_id               object
customer_id                  object
transaction_date             object
transaction_time             object
hour_of_day                   int64
is_weekend                    int64
is_night_transaction          int64
country                      object
city                         object
merchant_category            object
payment_method               object
device_type                  object
customer_age                  int64
credit_score                  int64
account_age_years           float64
account_balance             float64
transaction_amount          float64
num_prev_transactions         int64
transaction_freq_monthly      int64
distance_from_home_km       float64
time_since_last_txn_hrs     float64
is_international              int64
failed_attempts               int64
pin_changed_recently          int64
is_fraud                      int64
fraud_type                   object
dtype: object

In [8]:
df.shape

(1000000, 26)

In [9]:
df["transaction_date"].min(), df["transaction_date"].max()

('2020-01-01', '2024-12-30')

In [10]:
df["transaction_date"].is_monotonic_increasing

False

In [11]:
df[["transaction_date", "transaction_time"]].head(10)

,transaction_date,transaction_time
0,2023-08-17,21:13:00
1,2024-02-06,05:16:00
2,2024-06-28,12:15:00
3,2023-03-16,02:53:00
4,2024-07-12,12:39:00
5,2024-06-26,04:28:00
6,2020-09-05,19:22:00
7,2024-09-02,14:02:00
8,2021-12-07,03:23:00
9,2020-03-30,21:08:00


In [12]:
df["transaction_date"].nunique()

1826

In [13]:
df["transaction_time"].min(), df["transaction_time"].max()

('00:00:00', '23:59:00')

In [14]:
df[["transaction_date", "transaction_time"]].duplicated().sum()

np.int64(168015)

In [15]:
df["transaction_date"].head()

0    2023-08-17
1    2024-02-06
2    2024-06-28
3    2023-03-16
4    2024-07-12
Name: transaction_date, dtype: object

In [16]:
df["transaction_time"].head()

0    21:13:00
1    05:16:00
2    12:15:00
3    02:53:00
4    12:39:00
Name: transaction_time, dtype: object

In [17]:
(df["hour_of_day"] == df["transaction_time"].str[:2].astype(int)).value_counts()

True    1000000
Name: count, dtype: int64

In [18]:
(
    df["is_weekend"]
    == pd.to_datetime(df["transaction_date"]).dt.dayofweek.isin([5, 6]).astype(int)
).value_counts()

True    1000000
Name: count, dtype: int64

In [19]:
(
    df["is_night_transaction"]
    == df["transaction_time"].str[:2].astype(int).isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
).value_counts()

True     958570
False     41430
Name: count, dtype: int64

In [20]:
df.loc[
    df["is_night_transaction"]
    != df["transaction_time"].str[:2].astype(int).isin(
        [22, 23, 0, 1, 2, 3, 4, 5]
    ).astype(int),
    ["transaction_time", "hour_of_day", "is_night_transaction"]
].head(10)

,transaction_time,hour_of_day,is_night_transaction
20,06:03:00,6,1
42,06:58:00,6,1
56,06:15:00,6,1
146,06:11:00,6,1
154,06:45:00,6,1
162,06:46:00,6,1
227,06:16:00,6,1
282,06:39:00,6,1
290,06:47:00,6,1
295,06:03:00,6,1


In [21]:
# ============================================================
# STEP 4 — TEMPORAL / AVAILABILITY AUDIT
# Remaining consistency checks
# ============================================================

# Convert only for audit — original df remains unchanged
audit_date = pd.to_datetime(df["transaction_date"])

# 1. Weekend distribution
print("Weekend distribution:")
print(df["is_weekend"].value_counts().sort_index())

# 2. Night-transaction distribution
print("\nNight transaction distribution:")
print(df["is_night_transaction"].value_counts().sort_index())

# 3. International transaction distribution
print("\nInternational transaction distribution:")
print(df["is_international"].value_counts().sort_index())

# 4. Recent PIN change distribution
print("\nPIN changed recently distribution:")
print(df["pin_changed_recently"].value_counts().sort_index())

# 5. Failed attempts distribution
print("\nFailed attempts:")
print(df["failed_attempts"].describe())

# 6. Transaction amount distribution
print("\nTransaction amount:")
print(df["transaction_amount"].describe())

# 7. Time since last transaction
print("\nTime since last transaction:")
print(df["time_since_last_txn_hrs"].describe())

# 8. Date ordering check
print("\nChronological order:")
print(df["transaction_date"].is_monotonic_increasing)

# 9. Date range
print("\nDate range:")
print(audit_date.min(), "to", audit_date.max())

Weekend distribution:
is_weekend
0    713978
1    286022
Name: count, dtype: int64

Night transaction distribution:
is_night_transaction
0    624943
1    375057
Name: count, dtype: int64

International transaction distribution:
is_international
0    850006
1    149994
Name: count, dtype: int64

PIN changed recently distribution:
pin_changed_recently
0    919561
1     80439
Name: count, dtype: int64

Failed attempts:
count    1000000.000000
mean           0.380072
std            0.914603
min            0.000000
25%            0.000000
50%            0.000000
75%            0.000000
max            5.000000
Name: failed_attempts, dtype: float64

Transaction amount:
count    1000000.000000
mean         204.724665
std          459.567802
min            1.000000
25%           33.400000
50%           73.120000
75%          181.450000
max        46129.600000
Name: transaction_amount, dtype: float64

Time since last transaction:
count    1000000.000000
mean          12.000945
std           11.9

In [22]:
# ============================================================
# STEP 5 — LEAKAGE AUDIT
# Identifier and target-related fields
# ============================================================

leakage_candidates = [
    "transaction_id",
    "customer_id",
    "is_fraud",
    "fraud_type"
]

df[leakage_candidates].head()

,transaction_id,customer_id,is_fraud,fraud_type
0,TXN0000000001,CUST00121959,0,NaN
1,TXN0000000002,CUST00146868,0,NaN
2,TXN0000000003,CUST00131933,0,NaN
3,TXN0000000004,CUST00103695,1,Synthetic Identity
4,TXN0000000005,CUST00119880,0,NaN


In [23]:
print("transaction_id unique:", df["transaction_id"].nunique())
print("customer_id unique:", df["customer_id"].nunique())

print("\nfraud_type by target:")
print(pd.crosstab(
    df["is_fraud"],
    df["fraud_type"].notna()
))

transaction_id unique: 1000000
customer_id unique: 198662

fraud_type by target:
fraud_type   False  True 
is_fraud                 
0           944745      0
1                0  55255


In [24]:
# ============================================================
# STEP 5 — LEAKAGE AUDIT
# ============================================================

# Potential identifiers / target-derived fields
audit_cols = [
    "transaction_id",
    "customer_id",
    "fraud_type"
]

print("=" * 70)
print("IDENTIFIER / TARGET-DERIVED AUDIT")
print("=" * 70)

for col in audit_cols:
    print(f"\n{col}")
    print("-" * 40)
    print("Unique:", df[col].nunique(dropna=False))
    print("Missing:", df[col].isna().sum())

# ------------------------------------------------------------
# Target relationship of suspicious fields
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET RELATIONSHIP")
print("=" * 70)

for col in ["fraud_type"]:
    print(f"\n{col} vs is_fraud:")
    print(pd.crosstab(
        df["is_fraud"],
        df[col].notna(),
        margins=True
    ))

# ------------------------------------------------------------
# Fraud type distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FRAUD TYPE DISTRIBUTION")
print("=" * 70)

print(df["fraud_type"].value_counts(dropna=False))

# ------------------------------------------------------------
# Customer transaction frequency
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CUSTOMER ID REPEAT STRUCTURE")
print("=" * 70)

customer_txn_counts = df["customer_id"].value_counts()

print(customer_txn_counts.describe())

# ------------------------------------------------------------
# Potential temporal/post-event variables
# ------------------------------------------------------------

temporal_candidate_cols = [
    "transaction_date",
    "transaction_time",
    "hour_of_day",
    "is_weekend",
    "is_night_transaction",
    "num_prev_transactions",
    "transaction_freq_monthly",
    "time_since_last_txn_hrs",
    "failed_attempts",
    "pin_changed_recently"
]

print("\n" + "=" * 70)
print("TEMPORAL / AVAILABILITY CANDIDATES")
print("=" * 70)

print(temporal_candidate_cols)

# ------------------------------------------------------------
# Current leakage decision — documented, NOT applied yet
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PRELIMINARY LEAKAGE DECISION")
print("=" * 70)

print("Exclude: transaction_id")
print("Exclude: customer_id")
print("Exclude: fraud_type")
print("Keep for further availability audit: all remaining transaction features")

IDENTIFIER / TARGET-DERIVED AUDIT

transaction_id
----------------------------------------
Unique: 1000000
Missing: 0

customer_id
----------------------------------------
Unique: 198662
Missing: 0

fraud_type
----------------------------------------
Unique: 7
Missing: 944745

TARGET RELATIONSHIP

fraud_type vs is_fraud:
fraud_type   False   True      All
is_fraud                          
0           944745      0   944745
1                0  55255    55255
All         944745  55255  1000000

FRAUD TYPE DISTRIBUTION
fraud_type
NaN                   944745
Phishing                9252
Account Takeover        9232
Synthetic Identity      9214
Card Cloning            9200
Friendly Fraud          9187
Identity Theft          9170
Name: count, dtype: int64

CUSTOMER ID REPEAT STRUCTURE
count    198662.000000
mean          5.033675
std           2.201607
min           1.000000
25%           3.000000
50%           5.000000
75%           6.000000
max          19.000000
Name: count, dtype: flo

In [25]:
# ============================================================
# STEP 6 — FEATURE EXCLUSION & FINAL FEATURE SPACE
# ============================================================

TARGET = "is_fraud"

EXCLUDE_COLUMNS = [
    "transaction_id",   # unique transaction identifier
    "customer_id",      # customer identifier
    "fraud_type",       # target-derived information
    TARGET              # prediction target
]

# Create model feature matrix
X = df.drop(columns=EXCLUDE_COLUMNS).copy()

# Target
y = df[TARGET].copy()

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("=" * 70)
print("STEP 6 — FINAL FEATURE SPACE")
print("=" * 70)

print(f"\nOriginal columns : {df.shape[1]}")
print(f"Excluded columns : {len(EXCLUDE_COLUMNS)}")
print(f"Model features   : {X.shape[1]}")
print(f"Target size      : {y.shape[0]:,}")

print("\nExcluded columns:")
for col in EXCLUDE_COLUMNS:
    print(f"  - {col}")

print("\nFinal model features:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

STEP 6 — FINAL FEATURE SPACE

Original columns : 26
Excluded columns : 4
Model features   : 22
Target size      : 1,000,000

Excluded columns:
  - transaction_id
  - customer_id
  - fraud_type
  - is_fraud

Final model features:
['transaction_date', 'transaction_time', 'hour_of_day', 'is_weekend', 'is_night_transaction', 'country', 'city', 'merchant_category', 'payment_method', 'device_type', 'customer_age', 'credit_score', 'account_age_years', 'account_balance', 'transaction_amount', 'num_prev_transactions', 'transaction_freq_monthly', 'distance_from_home_km', 'time_since_last_txn_hrs', 'is_international', 'failed_attempts', 'pin_changed_recently']

Target distribution:
is_fraud
0    944745
1     55255
Name: count, dtype: int64


In [26]:
# ============================================================
# STEP 7 — TEMPORAL TRAIN / VALIDATION / TEST SPLIT
# 70% Train | 15% Validation | 15% Test
# ============================================================

# Create a temporary chronological timestamp
timestamp = pd.to_datetime(
    df["transaction_date"].astype(str) + " " +
    df["transaction_time"].astype(str)
)

# Sort ONLY for temporal splitting
temporal_order = timestamp.argsort(kind="mergesort")

df_temporal = df.iloc[temporal_order].reset_index(drop=True)

# ------------------------------------------------------------
# Split points
# ------------------------------------------------------------

n = len(df_temporal)

train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_df = df_temporal.iloc[:train_end].copy()
val_df   = df_temporal.iloc[train_end:val_end].copy()
test_df  = df_temporal.iloc[val_end:].copy()

# ------------------------------------------------------------
# Separate X and y
# ------------------------------------------------------------

X_train = train_df.drop(columns=EXCLUDE_COLUMNS)
y_train = train_df[TARGET]

X_val = val_df.drop(columns=EXCLUDE_COLUMNS)
y_val = val_df[TARGET]

X_test = test_df.drop(columns=EXCLUDE_COLUMNS)
y_test = test_df[TARGET]

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("=" * 70)
print("STEP 7 — TEMPORAL SPLIT")
print("=" * 70)

print("\nShapes:")
print("Train      :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

print("\nTarget distribution:")
print("\nTrain:")
print(y_train.value_counts())

print("\nValidation:")
print(y_val.value_counts())

print("\nTest:")
print(y_test.value_counts())

print("\nTemporal boundaries:")

print(
    "\nTrain:",
    train_df["transaction_date"].iloc[0],
    "→",
    train_df["transaction_date"].iloc[-1]
)

print(
    "Validation:",
    val_df["transaction_date"].iloc[0],
    "→",
    val_df["transaction_date"].iloc[-1]
)

print(
    "Test:",
    test_df["transaction_date"].iloc[0],
    "→",
    test_df["transaction_date"].iloc[-1]
)

STEP 7 — TEMPORAL SPLIT

Shapes:
Train      : (700000, 22)
Validation : (150000, 22)
Test       : (150000, 22)

Target distribution:

Train:
is_fraud
0    661368
1     38632
Name: count, dtype: int64

Validation:
is_fraud
0    141619
1      8381
Name: count, dtype: int64

Test:
is_fraud
0    141758
1      8242
Name: count, dtype: int64

Temporal boundaries:

Train: 2020-01-01 → 2023-06-30
Validation: 2023-06-30 → 2024-03-31
Test: 2024-03-31 → 2024-12-30


In [27]:
# ============================================================
# STEP 8 — TRAIN-ONLY PREPROCESSING
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# ------------------------------------------------------------
# 1. Convert date/time into temporal features
# ------------------------------------------------------------

def create_temporal_features(data):
    data = data.copy()

    dt = pd.to_datetime(
        data["transaction_date"].astype(str) + " " +
        data["transaction_time"].astype(str)
    )

    # Calendar / temporal information available at transaction time
    data["transaction_year"] = dt.dt.year
    data["transaction_month"] = dt.dt.month
    data["transaction_day"] = dt.dt.day
    data["transaction_dayofweek"] = dt.dt.dayofweek

    # Raw date/time columns are no longer needed
    data = data.drop(
        columns=["transaction_date", "transaction_time"]
    )

    return data


# Apply identical feature construction to each split
X_train_fe = create_temporal_features(X_train)
X_val_fe   = create_temporal_features(X_val)
X_test_fe  = create_temporal_features(X_test)


# ------------------------------------------------------------
# 2. Define categorical and numerical features
# ------------------------------------------------------------

categorical_features = [
    "country",
    "city",
    "merchant_category",
    "payment_method",
    "device_type"
]

numerical_features = [
    col for col in X_train_fe.columns
    if col not in categorical_features
]


# ------------------------------------------------------------
# 3. Train-only preprocessing pipeline
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        )
    ]
)


# ------------------------------------------------------------
# 4. FIT ONLY ON TRAINING DATA
# ------------------------------------------------------------

X_train_processed = preprocessor.fit_transform(X_train_fe)

# Validation and test are TRANSFORM ONLY
X_val_processed = preprocessor.transform(X_val_fe)
X_test_processed = preprocessor.transform(X_test_fe)


# ------------------------------------------------------------
# 5. Verification
# ------------------------------------------------------------

print("=" * 70)
print("STEP 8 — TRAIN-ONLY PREPROCESSING")
print("=" * 70)

print("\nBefore preprocessing:")
print("Train:", X_train_fe.shape)
print("Validation:", X_val_fe.shape)
print("Test:", X_test_fe.shape)

print("\nAfter preprocessing:")
print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

print("\nPreprocessor fitted ONLY on training data.")

STEP 8 — TRAIN-ONLY PREPROCESSING

Before preprocessing:
Train: (700000, 24)
Validation: (150000, 24)
Test: (150000, 24)

After preprocessing:
Train: (700000, 75)
Validation: (150000, 75)
Test: (150000, 75)

Categorical features:
['country', 'city', 'merchant_category', 'payment_method', 'device_type']

Numerical features:
['hour_of_day', 'is_weekend', 'is_night_transaction', 'customer_age', 'credit_score', 'account_age_years', 'account_balance', 'transaction_amount', 'num_prev_transactions', 'transaction_freq_monthly', 'distance_from_home_km', 'time_since_last_txn_hrs', 'is_international', 'failed_attempts', 'pin_changed_recently', 'transaction_year', 'transaction_month', 'transaction_day', 'transaction_dayofweek']

Preprocessor fitted ONLY on training data.


In [28]:
# ============================================================
# STEP 9 — BASELINE MODEL EVALUATION
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# Baseline model
# ------------------------------------------------------------

baseline_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight=None
)

# ------------------------------------------------------------
# Train ONLY on training data
# ------------------------------------------------------------

baseline_rf.fit(X_train_processed, y_train)

# ------------------------------------------------------------
# Validation prediction
# ------------------------------------------------------------

y_val_pred = baseline_rf.predict(X_val_processed)
y_val_prob = baseline_rf.predict_proba(X_val_processed)[:, 1]

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

baseline_results = {
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred, zero_division=0),
    "Recall": recall_score(y_val, y_val_pred, zero_division=0),
    "F1": f1_score(y_val, y_val_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_val, y_val_prob),
    "PR_AUC": average_precision_score(y_val, y_val_prob)
}

print("=" * 70)
print("STEP 9 — BASELINE RANDOM FOREST")
print("=" * 70)

for metric, value in baseline_results.items():
    print(f"{metric:12s}: {value:.6f}")

STEP 9 — BASELINE RANDOM FOREST
Accuracy    : 0.944127
Precision   : 0.000000
Recall      : 0.000000
F1          : 0.000000
ROC_AUC     : 0.703501
PR_AUC      : 0.112550


In [29]:
# ============================================================
# STEP 10 — MODEL DEVELOPMENT
# ADA BOOST
# ============================================================

from sklearn.ensemble import AdaBoostClassifier

ada_model = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)

ada_model.fit(X_train_processed, y_train)

y_val_pred_ada = ada_model.predict(X_val_processed)
y_val_prob_ada = ada_model.predict_proba(X_val_processed)[:, 1]

ada_results = {
    "Accuracy": accuracy_score(y_val, y_val_pred_ada),
    "Precision": precision_score(y_val, y_val_pred_ada, zero_division=0),
    "Recall": recall_score(y_val, y_val_pred_ada, zero_division=0),
    "F1": f1_score(y_val, y_val_pred_ada, zero_division=0),
    "ROC_AUC": roc_auc_score(y_val, y_val_prob_ada),
    "PR_AUC": average_precision_score(y_val, y_val_prob_ada)
}

print("=" * 70)
print("STEP 10 — ADA BOOST")
print("=" * 70)

for metric, value in ada_results.items():
    print(f"{metric:12s}: {value:.6f}")

STEP 10 — ADA BOOST
Accuracy    : 0.944127
Precision   : 0.000000
Recall      : 0.000000
F1          : 0.000000
ROC_AUC     : 0.715961
PR_AUC      : 0.118472


In [30]:
# ============================================================
# STEP 11 — MODEL DEVELOPMENT
# KNN
# ============================================================

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    n_jobs=-1
)

knn_model.fit(X_train_processed, y_train)

y_val_pred_knn = knn_model.predict(X_val_processed)
y_val_prob_knn = knn_model.predict_proba(X_val_processed)[:, 1]

knn_results = {
    "Accuracy": accuracy_score(y_val, y_val_pred_knn),
    "Precision": precision_score(y_val, y_val_pred_knn, zero_division=0),
    "Recall": recall_score(y_val, y_val_pred_knn, zero_division=0),
    "F1": f1_score(y_val, y_val_pred_knn, zero_division=0),
    "ROC_AUC": roc_auc_score(y_val, y_val_prob_knn),
    "PR_AUC": average_precision_score(y_val, y_val_prob_knn)
}

print("=" * 70)
print("STEP 11 — KNN")
print("=" * 70)

for metric, value in knn_results.items():
    print(f"{metric:12s}: {value:.6f}")

STEP 11 — KNN
Accuracy    : 0.941473
Precision   : 0.128731
Recall      : 0.008233
F1          : 0.015476
ROC_AUC     : 0.558758
PR_AUC      : 0.068500


In [31]:
# ============================================================
# STEP 12 — STACKING
# RF + AdaBoost + KNN
# 5-Fold OOF Stacking
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# Base models
# ------------------------------------------------------------

base_models = {
    "RF": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight=None
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.1,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        n_jobs=-1
    )
}

# ------------------------------------------------------------
# 5-fold OOF prediction matrix
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions = np.zeros(
    (X_train_processed.shape[0], len(base_models))
)

val_predictions = np.zeros(
    (X_val_processed.shape[0], len(base_models))
)

model_names = list(base_models.keys())

# ------------------------------------------------------------
# Generate OOF predictions
# ------------------------------------------------------------

for model_idx, (name, model) in enumerate(base_models.items()):

    print(f"\nTraining {name}...")

    for fold, (train_idx, oof_idx) in enumerate(
        skf.split(X_train_processed, y_train), 1
    ):

        fold_model = clone(model)

        fold_model.fit(
            X_train_processed[train_idx],
            y_train.iloc[train_idx]
        )

        oof_predictions[oof_idx, model_idx] = (
            fold_model.predict_proba(
                X_train_processed[oof_idx]
            )[:, 1]
        )

        print(f"  Fold {fold}/5 complete")

    # Fit base model on full training data
    full_model = clone(model)

    full_model.fit(
        X_train_processed,
        y_train
    )

    val_predictions[:, model_idx] = (
        full_model.predict_proba(
            X_val_processed
        )[:, 1]
    )

# ------------------------------------------------------------
# Meta-learner
# ------------------------------------------------------------

meta_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

meta_model.fit(
    oof_predictions,
    y_train
)

# ------------------------------------------------------------
# Validation prediction
# ------------------------------------------------------------

y_val_prob_stack = meta_model.predict_proba(
    val_predictions
)[:, 1]

y_val_pred_stack = (
    y_val_prob_stack >= 0.50
).astype(int)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

stacking_results = {
    "Accuracy": accuracy_score(
        y_val, y_val_pred_stack
    ),
    "Precision": precision_score(
        y_val, y_val_pred_stack,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val, y_val_pred_stack,
        zero_division=0
    ),
    "F1": f1_score(
        y_val, y_val_pred_stack,
        zero_division=0
    ),
    "ROC_AUC": roc_auc_score(
        y_val, y_val_prob_stack
    ),
    "PR_AUC": average_precision_score(
        y_val, y_val_prob_stack
    )
}

print("\n" + "=" * 70)
print("STEP 12 — STACKING")
print("=" * 70)

print("\nOOF matrix:", oof_predictions.shape)
print("Validation meta-features:", val_predictions.shape)

print("\nStacking Results:")

for metric, value in stacking_results.items():
    print(f"{metric:12s}: {value:.6f}")


Training RF...
  Fold 1/5 complete
  Fold 2/5 complete
  Fold 3/5 complete
  Fold 4/5 complete
  Fold 5/5 complete

Training AdaBoost...
  Fold 1/5 complete
  Fold 2/5 complete
  Fold 3/5 complete
  Fold 4/5 complete
  Fold 5/5 complete

Training KNN...
  Fold 1/5 complete
  Fold 2/5 complete
  Fold 3/5 complete
  Fold 4/5 complete
  Fold 5/5 complete

STEP 12 — STACKING

OOF matrix: (700000, 3)
Validation meta-features: (150000, 3)

Stacking Results:
Accuracy    : 0.944067
Precision   : 0.333333
Recall      : 0.001074
F1          : 0.002141
ROC_AUC     : 0.717651
PR_AUC      : 0.122996


In [32]:
# ============================================================
# STEP 13 — MODEL DEVELOPMENT
# MLP
# ============================================================

from sklearn.neural_network import MLPClassifier

mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=2048,
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=5,
    random_state=42
)

# ------------------------------------------------------------
# Train ONLY on training data
# ------------------------------------------------------------

mlp_model.fit(
    X_train_processed,
    y_train
)

# ------------------------------------------------------------
# Validation prediction
# ------------------------------------------------------------

y_val_pred_mlp = mlp_model.predict(
    X_val_processed
)

y_val_prob_mlp = mlp_model.predict_proba(
    X_val_processed
)[:, 1]

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

mlp_results = {
    "Accuracy": accuracy_score(
        y_val,
        y_val_pred_mlp
    ),
    "Precision": precision_score(
        y_val,
        y_val_pred_mlp,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val,
        y_val_pred_mlp,
        zero_division=0
    ),
    "F1": f1_score(
        y_val,
        y_val_pred_mlp,
        zero_division=0
    ),
    "ROC_AUC": roc_auc_score(
        y_val,
        y_val_prob_mlp
    ),
    "PR_AUC": average_precision_score(
        y_val,
        y_val_prob_mlp
    )
}

print("=" * 70)
print("STEP 13 — MLP")
print("=" * 70)

for metric, value in mlp_results.items():
    print(f"{metric:12s}: {value:.6f}")

STEP 13 — MLP
Accuracy    : 0.944127
Precision   : 0.000000
Recall      : 0.000000
F1          : 0.000000
ROC_AUC     : 0.699333
PR_AUC      : 0.111190


In [33]:
# ============================================================
# STEP 14 — MODEL DEVELOPMENT
# LIGHTGBM
# ============================================================

from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# ------------------------------------------------------------
# Train ONLY on training data
# ------------------------------------------------------------

lgbm_model.fit(
    X_train_processed,
    y_train
)

# ------------------------------------------------------------
# Validation prediction
# ------------------------------------------------------------

y_val_pred_lgbm = lgbm_model.predict(
    X_val_processed
)

y_val_prob_lgbm = lgbm_model.predict_proba(
    X_val_processed
)[:, 1]

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

lgbm_results = {
    "Accuracy": accuracy_score(
        y_val,
        y_val_pred_lgbm
    ),
    "Precision": precision_score(
        y_val,
        y_val_pred_lgbm,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val,
        y_val_pred_lgbm,
        zero_division=0
    ),
    "F1": f1_score(
        y_val,
        y_val_pred_lgbm,
        zero_division=0
    ),
    "ROC_AUC": roc_auc_score(
        y_val,
        y_val_prob_lgbm
    ),
    "PR_AUC": average_precision_score(
        y_val,
        y_val_prob_lgbm
    )
}

print("=" * 70)
print("STEP 14 — LIGHTGBM")
print("=" * 70)

for metric, value in lgbm_results.items():
    print(f"{metric:12s}: {value:.6f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


STEP 14 — LIGHTGBM
Accuracy    : 0.944127
Precision   : 0.000000
Recall      : 0.000000
F1          : 0.000000
ROC_AUC     : 0.724500
PR_AUC      : 0.126279


In [34]:
# ============================================================
# STEP 15-A — BASELINE CLASS IMBALANCE ANALYSIS
# ============================================================

print("=" * 70)
print("STEP 15-A — BASELINE CLASS IMBALANCE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Class counts
# ------------------------------------------------------------

class_counts = y_train.value_counts().sort_index()

non_fraud = class_counts[0]
fraud = class_counts[1]

# ------------------------------------------------------------
# Class percentages
# ------------------------------------------------------------

class_percentages = (
    y_train.value_counts(normalize=True)
    .sort_index() * 100
)

# ------------------------------------------------------------
# Imbalance ratio
# ------------------------------------------------------------

imbalance_ratio = non_fraud / fraud

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\nTraining set:")
print("Total samples :", len(y_train))

print("\nClass distribution:")
print(class_counts)

print("\nClass percentage:")
print(class_percentages.round(4))

print("\nNon-fraud samples :", non_fraud)
print("Fraud samples     :", fraud)

print(
    "\nFraud rate        :",
    f"{class_percentages[1]:.4f}%"
)

print(
    "Non-fraud rate    :",
    f"{class_percentages[0]:.4f}%"
)

print(
    "\nImbalance ratio (Non-fraud : Fraud) :",
    f"{imbalance_ratio:.2f} : 1"
)

print("\nValidation set remains untouched.")
print("Test set remains untouched.")

STEP 15-A — BASELINE CLASS IMBALANCE ANALYSIS

Training set:
Total samples : 700000

Class distribution:
is_fraud
0    661368
1     38632
Name: count, dtype: int64

Class percentage:
is_fraud
0    94.4811
1     5.5189
Name: proportion, dtype: float64

Non-fraud samples : 661368
Fraud samples     : 38632

Fraud rate        : 5.5189%
Non-fraud rate    : 94.4811%

Imbalance ratio (Non-fraud : Fraud) : 17.12 : 1

Validation set remains untouched.
Test set remains untouched.


In [35]:
# ============================================================
# STEP 15-B — RANDOM UNDER-SAMPLING (RUS)
# ============================================================

from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(
    sampling_strategy=1.0,
    random_state=42
)

# RUS ONLY on training data
X_train_rus, y_train_rus = rus.fit_resample(
    X_train_processed,
    y_train
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("=" * 70)
print("STEP 15-B — RANDOM UNDER-SAMPLING")
print("=" * 70)

print("\nOriginal training shape:")
print(X_train_processed.shape)

print("\nAfter RUS:")
print(X_train_rus.shape)

print("\nOriginal target distribution:")
print(y_train.value_counts())

print("\nRUS target distribution:")
print(y_train_rus.value_counts())

print("\nValidation set: UNTOUCHED")
print("Test set      : UNTOUCHED")

STEP 15-B — RANDOM UNDER-SAMPLING

Original training shape:
(700000, 75)

After RUS:
(77264, 75)

Original target distribution:
is_fraud
0    661368
1     38632
Name: count, dtype: int64

RUS target distribution:
is_fraud
0    38632
1    38632
Name: count, dtype: int64

Validation set: UNTOUCHED
Test set      : UNTOUCHED


In [36]:
# ============================================================
# STEP 15-C — SMOTE
# ============================================================

from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy=1.0,
    random_state=42,
    k_neighbors=5
)

# SMOTE ONLY on training data
X_train_smote, y_train_smote = smote.fit_resample(
    X_train_processed,
    y_train
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("=" * 70)
print("STEP 15-C — SMOTE")
print("=" * 70)

print("\nOriginal training shape:")
print(X_train_processed.shape)

print("\nAfter SMOTE:")
print(X_train_smote.shape)

print("\nOriginal target distribution:")
print(y_train.value_counts())

print("\nSMOTE target distribution:")
print(y_train_smote.value_counts())

print("\nValidation set: UNTOUCHED")
print("Test set      : UNTOUCHED")

STEP 15-C — SMOTE

Original training shape:
(700000, 75)

After SMOTE:
(1322736, 75)

Original target distribution:
is_fraud
0    661368
1     38632
Name: count, dtype: int64

SMOTE target distribution:
is_fraud
0    661368
1    661368
Name: count, dtype: int64

Validation set: UNTOUCHED
Test set      : UNTOUCHED


In [37]:
# ============================================================
# STEP 15-D — SMOTE-ENN
# OPTIMIZED IMPLEMENTATION
# ============================================================

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours

# ------------------------------------------------------------
# 1. SMOTE component
# ------------------------------------------------------------

smote_component = SMOTE(
    sampling_strategy=1.0,
    k_neighbors=5,
    random_state=42
)

# ------------------------------------------------------------
# 2. ENN component
# ------------------------------------------------------------

enn_component = EditedNearestNeighbours(
    n_neighbors=3,
    kind_sel="all",
    n_jobs=-1
)

# ------------------------------------------------------------
# 3. SMOTE-ENN
# ------------------------------------------------------------

smote_enn = SMOTEENN(
    sampling_strategy=1.0,
    random_state=42,
    smote=smote_component,
    enn=enn_component
)

# ------------------------------------------------------------
# 4. Apply ONLY to training data
# ------------------------------------------------------------

X_train_smote_enn, y_train_smote_enn = (
    smote_enn.fit_resample(
        X_train_processed,
        y_train
    )
)

# ------------------------------------------------------------
# 5. Verification
# ------------------------------------------------------------

print("=" * 70)
print("STEP 15-D — SMOTE-ENN")
print("=" * 70)

print("\nOriginal training shape:")
print(X_train_processed.shape)

print("\nAfter SMOTE-ENN:")
print(X_train_smote_enn.shape)

print("\nOriginal target distribution:")
print(y_train.value_counts())

print("\nSMOTE-ENN target distribution:")
print(y_train_smote_enn.value_counts())

print("\nValidation set: UNTOUCHED")
print("Test set      : UNTOUCHED")

STEP 15-D — SMOTE-ENN

Original training shape:
(700000, 75)

After SMOTE-ENN:
(1322735, 75)

Original target distribution:
is_fraud
0    661368
1     38632
Name: count, dtype: int64

SMOTE-ENN target distribution:
is_fraud
0    661368
1    661367
Name: count, dtype: int64

Validation set: UNTOUCHED
Test set      : UNTOUCHED


In [38]:
# ============================================================
# STEP 16 — CANDIDATE COMPARISON
# RUS vs SMOTE vs SMOTE-ENN
# LightGBM on each training configuration
# ============================================================

from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
import pandas as pd

# ------------------------------------------------------------
# 1. Candidate training datasets
# ------------------------------------------------------------

candidate_datasets = {
    "RUS": (
        X_train_rus,
        y_train_rus
    ),
    "SMOTE": (
        X_train_smote,
        y_train_smote
    ),
    "SMOTE-ENN": (
        X_train_smote_enn,
        y_train_smote_enn
    )
}

comparison_results = []

# ------------------------------------------------------------
# 2. Train and evaluate each candidate
# ------------------------------------------------------------

for name, (X_resampled, y_resampled) in candidate_datasets.items():

    print("\n" + "=" * 70)
    print(f"Training LightGBM with {name}")
    print("=" * 70)

    model = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    # Train ONLY on the selected resampled training data
    model.fit(
        X_resampled,
        y_resampled
    )

    # Validation prediction
    y_pred = model.predict(
        X_val_processed
    )

    y_prob = model.predict_proba(
        X_val_processed
    )[:, 1]

    # Metrics
    result = {
        "Method": name,
        "Train_Size": len(y_resampled),
        "Accuracy": accuracy_score(
            y_val,
            y_pred
        ),
        "Precision": precision_score(
            y_val,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_val,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_val,
            y_pred,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_val,
            y_prob
        ),
        "PR_AUC": average_precision_score(
            y_val,
            y_prob
        )
    }

    comparison_results.append(result)

    print("\nResults:")
    print(f"Accuracy    : {result['Accuracy']:.6f}")
    print(f"Precision   : {result['Precision']:.6f}")
    print(f"Recall      : {result['Recall']:.6f}")
    print(f"F1          : {result['F1']:.6f}")
    print(f"ROC_AUC     : {result['ROC_AUC']:.6f}")
    print(f"PR_AUC      : {result['PR_AUC']:.6f}")


# ------------------------------------------------------------
# 3. Final comparison table
# ------------------------------------------------------------

candidate_comparison = pd.DataFrame(
    comparison_results
)

candidate_comparison = candidate_comparison.sort_values(
    by="PR_AUC",
    ascending=False
).reset_index(drop=True)

print("\n")
print("=" * 70)
print("STEP 16 — CANDIDATE COMPARISON")
print("=" * 70)

print(
    candidate_comparison.to_string(
        index=False
    )
)

print("\nValidation set: UNTOUCHED")
print("Test set      : UNTOUCHED")


Training LightGBM with RUS


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Results:
Accuracy    : 0.652793
Precision   : 0.101525
Recall      : 0.664241
F1          : 0.176129
ROC_AUC     : 0.721673
PR_AUC      : 0.123457

Training LightGBM with SMOTE


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Results:
Accuracy    : 0.944127
Precision   : 0.000000
Recall      : 0.000000
F1          : 0.000000
ROC_AUC     : 0.722397
PR_AUC      : 0.125352

Training LightGBM with SMOTE-ENN


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Results:
Accuracy    : 0.944127
Precision   : 0.000000
Recall      : 0.000000
F1          : 0.000000
ROC_AUC     : 0.722226
PR_AUC      : 0.124542


STEP 16 — CANDIDATE COMPARISON
   Method  Train_Size  Accuracy  Precision   Recall       F1  ROC_AUC   PR_AUC
    SMOTE     1322736  0.944127   0.000000 0.000000 0.000000 0.722397 0.125352
SMOTE-ENN     1322735  0.944127   0.000000 0.000000 0.000000 0.722226 0.124542
      RUS       77264  0.652793   0.101525 0.664241 0.176129 0.721673 0.123457

Validation set: UNTOUCHED
Test set      : UNTOUCHED
